# BRONZE_DOMINIO

**Origem:** tabelas de conversão oficiais do TabWin/DATASUS, que traduzem os códigos dos
arquivos para descrições:

| TABELA | Arquivo | Pacote |
|---|---|---|
| REGIAO_SAUDE | `AL_regsaud.cnv` | TAB_SIA.zip |
| MACRORREGIAO | `AL_macsaud.cnv` | TAB_SIA.zip |
| TIPO_UNIDADE | `TP_ESTAB.CNV` | TAB_SIA.zip |
| COMPLEXIDADE | `COMPLEX.CNV` | TAB_SIA.zip |
| CARATER | `CARATER.CNV` | TAB_SIA.zip |
| GRUPO_PROCEDIMENTO | `TB_GRUPO.DBF` | TAB_SIA.zip |
| ESPECIALIDADE | `LEITOS.CNV` | TAB_SIH.zip |

Usar as tabelas oficiais evita listas digitadas à mão no código.

**Estratégia de carga:** full (`TRUNCATE` + `INSERT`).

In [ ]:
import sys
sys.path.append("..")
from dw_utils import baixar_tab_sia, baixar_zip, TAB_SIH, ler_cnv, substituir, consultar
import os
import tempfile
from dbfread import DBF

In [ ]:
CNV_SIA = {
    "REGIAO_SAUDE": "AL_REGSAUD.CNV",
    "MACRORREGIAO": "AL_MACSAUD.CNV",
    "TIPO_UNIDADE": "TP_ESTAB.CNV",
    "COMPLEXIDADE": "COMPLEX.CNV",
    "CARATER": "CARATER.CNV",
}

with tempfile.TemporaryDirectory() as pasta:
    sia = baixar_tab_sia(os.path.join(pasta, "sia"))
    sih = baixar_zip(TAB_SIH, os.path.join(pasta, "sih"))

    linhas = [(tabela, cod, desc) for tabela, arq in CNV_SIA.items() for cod, desc in ler_cnv(sia[arq])]
    linhas += [("ESPECIALIDADE", cod, desc) for cod, desc in ler_cnv(sih["LEITOS.CNV"])]
    linhas += [("GRUPO_PROCEDIMENTO", r["CO_GRUPO"], r["NO_GRUPO"])
               for r in DBF(sia["TB_GRUPO.DBF"], encoding="iso-8859-1")]

substituir("BRONZE_DOMINIO", ["TABELA", "CODIGO", "DESCRICAO"], linhas)

In [ ]:
consultar("SELECT TABELA, COUNT(*) AS LINHAS, COUNT(DISTINCT CODIGO) AS CODIGOS FROM BRONZE_DOMINIO GROUP BY TABELA ORDER BY 1")